# Lesson 7 : Agent Skills

Agent Skill is introduced by Anthropic on October 2025 and now it's an open (and cross-platform) project. (See [here](https://agentskills.io/home) for Agent Skill's specification.)  
The objective of Agent Skills is to enable highly modularization and reuse of actions and knowledge - such as, instructions, tool usage, and knowledge retrieval. Unlike tools in agents, Agent Skills can encapsulate even large, complex tasks and its assets into a single skill, allowing downstream processes to consume only the final result rather than the entire intermediate context. This also helps reduce token usage and improves overall efficiency.

In Microsoft Agent Framework, you can use the following 3 types of skill's definition.

- File-based skills
- Class-based skills
- Code-defined skills

In this exercise, we create brief custom skills (a file-based skill and a code-defined skill) to create a expense report in Microsoft Agent Framework.

## Create a file-based skill

There exist a lot of pre-built (reusable) existing skills (see [here](https://github.com/heilcheng/awesome-agent-skills)), but in this exercise, we briefly build our own custom file skill to create a unique expense report for our virtual company as follows.

First we create a file (named ```SKILL.md```) to describe skill's body as follows.  
Please note that the ```%%writefile``` directive (the first line in the following code) in Jupyter notebook cell indicates that this code is stored as file, not executed here. (The same applies in the following cells.)

In [1]:
import os
skill_folder = path = os.path.join("skills", "corporate-expense-report")
os.makedirs(skill_folder, exist_ok=True)

In [2]:
%%writefile skills/corporate-expense-report/SKILL.md
---
name: corporate-expense-report
description: This skill calculates additional fee in each expense item and create a corporate expense report using template.
---

# Corporate Expense Report Skill

This skill provides instructions on how to create a corporate expense report.

## Capabilities

- Infer the category from the description in each expense item.  
  Available categories: "international transport", "domestic transport", "meal", and "misc"
- Calculate additional fee (tax + transaction fee) for each item
- Generate a expense report including additional fee

## Input Format

Provide the list of expense - in which each item includes description, date, and amount.

## Output Format

Show expense report using template: [assets/expense_template.md](assets/expense_template.md)

## Additional Fee

Additional fee in each item consists of tax fee and transaction fee.  
All of these costs are calculated by multiplying the expenses by a certain multiplier, and the multipliers in each category are listed in table on [assets/additional_fees.md](assets/additional_fees.md).

## Policy check

Use contoso-expense-policy skill to verify whether each item complies with company policy, and display the result as "OK" or "NG" in the output.

## Total fee

Total fee is the total for items with "OK" policy status.

Writing skills/corporate-expense-report/SKILL.md


In above skill's body, we refer assets - ```expense_template.md``` and ```additional_fees.md``` - which include output template for expense report and a list of tax/transaction ratio, respectively.  
Now we create these assets as follows.

In [3]:
asset_folder = path = os.path.join(skill_folder, "assets")
os.makedirs(asset_folder, exist_ok=True)

In [4]:
%%writefile skills/corporate-expense-report/assets/expense_template.md
| Date | Category | Amount (USD) | Additional fee (USD) | Policy check | Sub total (USD) |
|------|----------|--------------|----------------------|--------------|-----------------|
|      |          |              |                      |              |                 |

Total: [total fee]

Writing skills/corporate-expense-report/assets/expense_template.md


In [5]:
%%writefile skills/corporate-expense-report/assets/additional_fees.md
| Category                | Tax percentage | Transaction percentage |
|-------------------------|----------------|------------------------|
| international transport | 0 %            | 10 %                   |
| domestic transport      | 0 %            | 5 %                    |
| meal                    | 10 %           | 0 %                    |
| misc                    | 3 %            | 0 %                    |

Writing skills/corporate-expense-report/assets/additional_fees.md


The ```skills``` folder will then have the following structure.  
In this exercise, we use only a single file skill to solve the problem, but you can include a lot of existing skills in ```skills``` folder.

```
skills/
└── corporate-expense-report/
    ├── SKILL.md
    └── assets/
        ├── expense_template.md
        └── additional_fees.md
```

## Create a code-defined skill

Sometimes we need a resource to execute in-process logic code, rather than static text.  
In such cases, we have used in-process local function tools, MCP tool calling, or code interpreter tools in previous examples.  
On skill's framework in Microsoft Agent Framework, we can use code-defined skills to tackle with these scenarios.

In this example, we define a code-defined tool to check whether the expense complies with the company policy.

> Note : You can also create code (.py) for large tasks (such as, conversion, creation, ...) and register them as skill's assets in file-based skills.  
> Unlike code-defined skills, these tasks are executed in sandbox execution context, not in-process. (Python packages for task execution will also be installed as needed.)

In [6]:
from agent_framework import InlineSkill, SkillFrontmatter

contoso_policy_skill = InlineSkill(
    frontmatter=SkillFrontmatter(
        name="contoso-expense-policy",
        description="This skill is used for processing tasks related to expense policy in Contoso company.",
    ),
    instructions="Use this skill when some task related to expense policy in Contoso company is required.",
)

@contoso_policy_skill.script(
    name="check-policy",
    description="Check whether the expense complies with the policy",
)
def check_expense_policy(expense_category: str, expense_description: str) -> bool:
    """Check whether the expense complies with the policy. 

    Args:
        expense_category: "international transport", "domestic transport", "meal", or "misc"
        expense_description: description of the expense details

    Returns:
        True: OK
        False: NG
    """
    if expense_category.lower() == "misc":
        if "souvenir" in expense_description.lower():
            return False
    return True

## Run agent with skills

Now let's build an agent to use above skills - a file-based skill ("expense" skill) and a code-defined skill ("utilities" skill).

First we initilize the client object as usual.

In [7]:
from dotenv import load_dotenv
from agent_framework.foundry import FoundryChatClient
from azure.identity.aio import AzureCliCredential

load_dotenv()

credential = AzureCliCredential()
client = FoundryChatClient(credential=credential)

Now we define a skill provider, which refers to above skill's folder (which includes a file-based "expense" skill) and a code-defined skill.

In [8]:
from agent_framework import (
    AggregatingSkillsSource,
    DeduplicatingSkillsSource,
    FileSkillsSource,
    InMemorySkillsSource,
    SkillsProvider,
)

skills_provider = SkillsProvider(
    DeduplicatingSkillsSource(
        AggregatingSkillsSource([
            FileSkillsSource("skills"),
            InMemorySkillsSource([contoso_policy_skill]),
        ])
    )
)

Now we create an agent with this skill's provider as follows.  
Same as Lesson 5, we set a provider in ```context_providers``` property in the agent.

> Note : Every tool used in ```SkillsProvider``` is registered with ```approval_mode="always_require"```, so each skill operation needs approval. To run unattended (automatically approve all tools), I have set ```ToolApprovalMiddleware``` as follows.

In [9]:
from agent_framework import Agent, ToolApprovalMiddleware

agent = Agent(
    name="AgentWithSkills",
    client=client,
    instructions="You are a helpful assistant.",
    context_providers=[skills_provider],
    middleware=[ToolApprovalMiddleware(auto_approval_rules=[SkillsProvider.all_tools_auto_approval_rule])],
)

Now let's run the agent.  
In this skill, the following ration of tax and transaction fee should be added in each item, and the agent shows an expense report with a table which has columns - "data", "category", "amount", "additional fee", and "sub total".

| category | tax ratio | transaction fee ratio |
|----------|-----------|-----------------------|
| international transport | 0.00 | 0.10 |
| domestic transport | 0.00 | 0.05 |
| meal | 0.10 | 0.00 |
| misc | 0.03 | 0.00 |

In [10]:
from IPython.display import Markdown, display

prompt = """Return an expense report of the following.

- flight from JFK to SEA (round trip)
    - date : 2026/02/09
    - amount : 2800
- dinner
    - date : 2026/02/09
    - amount : 80
- lunch
    - date : 2026/02/10
    - amount : 37
- souvenirs
    - date : 2026/02/10
    - amount : 40
"""

session = agent.create_session()
result = await agent.run(
    prompt,
    session=session,
)
display(Markdown(result.text))

| Date       | Category           | Amount (USD) | Additional fee (USD) | Policy check | Sub total (USD) |
|------------|--------------------|--------------|----------------------|--------------|-----------------|
| 2026/02/09 | domestic transport | 2800.00      | 140.00               | OK           | 2940.00         |
| 2026/02/09 | meal               | 80.00        | 8.00                 | OK           | 88.00           |
| 2026/02/10 | meal               | 37.00        | 3.70                 | OK           | 40.70           |
| 2026/02/10 | misc               | 40.00        | 1.20                 | NG           | 41.20           |

Total: 3068.70

## [Note] Foundry skills (server-side skill definitions)

You can register agent skills in Microsoft Foundry and deliver them to the local agent by the following 2 ways :

- attach to a Foundry toolbox so any MCP client can discover and load them alongside tools (The agent discovers and loads these skills through MCP resources.)
- download directly into the local agent for direct injection into each context

> Note : Foundry skills (server-side skills) will improve reusability on building your agents.

For implementing the former pattern with Microsoft Agent Framework (MAF), please refer to [here](https://github.com/microsoft/agent-framework/tree/main/python/samples/02-agents/skills/mcp_based_skill)